# Módulo 9 · Clase 6 — API REST con FastAPI

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Anatomía de una API REST |
| 2 | FastAPI: app, rutas, Pydantic |
| 3 | Endpoint `POST /predict` |
| 4 | Lanzar el servidor con uvicorn |
| 5 | Probar la API con `requests` |

In [ ]:
# Instalación (si no están)
# !pip install fastapi uvicorn[standard] pydantic requests

print("Librerías necesarias:")
import subprocess, sys
paquetes = ['fastapi', 'uvicorn', 'pydantic', 'requests']
for p in paquetes:
    resultado = subprocess.run([sys.executable, '-c', f'import {p}; print({p}.__version__)'],
                               capture_output=True, text=True)
    version = resultado.stdout.strip() if resultado.returncode == 0 else 'NO INSTALADO'
    print(f"  {p:15}: {version}")

---
## 1. Anatomía de una API REST

```
Cliente                     Servidor (FastAPI)
-------                     ------------------
POST /predict               @app.post("/predict")
Content-Type: application/json
Body: {"alcohol": 12.5, ...}   → Pydantic valida
                               → modelo.predict()
                            ← 200 OK
                            ← {"prediccion": "bueno", ...}
```

**Verbos HTTP más usados en ML APIs:**

| Verbo | Uso |
|-------|-----|
| `GET`    | Leer información (health check, metadatos del modelo) |
| `POST`   | Enviar datos para obtener predicción |
| `DELETE` | Eliminar un registro (predicción guardada) |

**Códigos de estado HTTP clave:**
- `200 OK` — todo bien
- `422 Unprocessable Entity` — datos de entrada inválidos (Pydantic)
- `500 Internal Server Error` — error en el servidor

---
## 2. Crear la aplicación FastAPI

In [ ]:
%%writefile api/main.py
"""
API REST para predicción de calidad de vinos.
Modelos entrenados en M9_C4 (ML) y M9_C5 (DL).
"""
from pathlib import Path
import json
import numpy as np
import joblib
from datetime import datetime
from typing import Optional, Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, model_validator

app = FastAPI(
    title="Wine Quality API",
    description="Predice si un vino es de buena calidad y su puntuación estimada.",
    version="1.0.0"
)

# ---------- Modelos Pydantic ----------

class VinoEntrada(BaseModel):
    """Características fisicoquímicas de un vino."""
    tipo:                  Literal['tinto', 'blanco']
    fixed_acidity:         float = Field(..., gt=0, le=20,  description="Acidez fija (g/L)")
    volatile_acidity:      float = Field(..., ge=0, le=2,   description="Acidez volátil (g/L)")
    citric_acid:           float = Field(..., ge=0, le=1,   description="Ácido cítrico (g/L)")
    residual_sugar:        float = Field(..., ge=0, le=65,  description="Azúcar residual (g/L)")
    chlorides:             float = Field(..., ge=0, le=1,   description="Cloruros (g/L)")
    free_sulfur_dioxide:   float = Field(..., ge=0, le=300, description="SO2 libre (mg/L)")
    total_sulfur_dioxide:  float = Field(..., ge=0, le=500, description="SO2 total (mg/L)")
    density:               float = Field(..., ge=0.98, le=1.05, description="Densidad (g/cm³)")
    pH:                    float = Field(..., ge=2.5, le=4.5,   description="pH")
    sulphates:             float = Field(..., ge=0, le=2,        description="Sulfatos (g/L)")
    alcohol:               float = Field(..., ge=8, le=15,       description="Alcohol (%vol)")

    @model_validator(mode='after')
    def validar_so2(self):
        if self.free_sulfur_dioxide > self.total_sulfur_dioxide:
            raise ValueError("free_sulfur_dioxide no puede superar total_sulfur_dioxide")
        return self

    class Config:
        json_schema_extra = {
            "example": {
                "tipo": "tinto", "fixed_acidity": 7.4, "volatile_acidity": 0.7,
                "citric_acid": 0.0, "residual_sugar": 1.9, "chlorides": 0.076,
                "free_sulfur_dioxide": 11.0, "total_sulfur_dioxide": 34.0,
                "density": 0.9978, "pH": 3.51, "sulphates": 0.56, "alcohol": 9.4
            }
        }

class PrediccionSalida(BaseModel):
    prediccion:          str
    probabilidad:        float
    puntuacion_estimada: Optional[float] = None
    modelo:              str
    timestamp:           str

# ---------- Carga de modelos (lazy) ----------

_modelos = {}

FEATURES_ORDEN = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
]

def cargar_modelo_rf():
    if 'rf' not in _modelos:
        ruta = Path('models/rf_vinos_v1.joblib')
        if not ruta.exists():
            raise HTTPException(500, "Modelo RF no encontrado. Ejecuta primero M9_C4.")
        _modelos['rf'] = joblib.load(ruta)
    return _modelos['rf']

def entrada_a_array(vino: VinoEntrada) -> np.ndarray:
    """Convierte el modelo Pydantic a array con features derivadas."""
    d = {
        'fixed acidity':        vino.fixed_acidity,
        'volatile acidity':     vino.volatile_acidity,
        'citric acid':          vino.citric_acid,
        'residual sugar':       vino.residual_sugar,
        'chlorides':            vino.chlorides,
        'free sulfur dioxide':  vino.free_sulfur_dioxide,
        'total sulfur dioxide': vino.total_sulfur_dioxide,
        'density':              vino.density,
        'pH':                   vino.pH,
        'sulphates':            vino.sulphates,
        'alcohol':              vino.alcohol,
    }
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if vino.tipo == 'tinto' else 0
    return np.array([[d[f] for f in FEATURES_ORDEN]])

# ---------- Rutas ----------

@app.get("/")
def raiz():
    return {"mensaje": "Wine Quality API v1.0", "docs": "/docs"}

@app.get("/health")
def health():
    return {"status": "ok", "timestamp": datetime.now().isoformat()}

@app.get("/model/info")
def info_modelo():
    meta_path = Path('models/rf_vinos_v1_meta.json')
    if not meta_path.exists():
        raise HTTPException(404, "Metadatos no encontrados")
    with open(meta_path, encoding='utf-8') as f:
        return json.load(f)

@app.post("/predict", response_model=PrediccionSalida)
def predecir(vino: VinoEntrada):
    """Predice si un vino es de buena calidad (quality >= 7)."""
    modelo = cargar_modelo_rf()
    X = entrada_a_array(vino)

    pred = int(modelo.predict(X)[0])
    prob = float(modelo.predict_proba(X)[0, 1])

    return PrediccionSalida(
        prediccion          = 'bueno' if pred == 1 else 'normal',
        probabilidad        = round(prob, 4),
        puntuacion_estimada = None,  # se añade en el ejercicio con el modelo DL
        modelo              = 'rf_v1',
        timestamp           = datetime.now().isoformat()
    )

@app.post("/predict/batch")
def predecir_batch(vinos: list[VinoEntrada]):
    """Predice para una lista de vinos (máximo 100 a la vez)."""
    if len(vinos) > 100:
        raise HTTPException(400, "Máximo 100 vinos por petición")
    modelo = cargar_modelo_rf()
    X = np.vstack([entrada_a_array(v) for v in vinos])
    preds = modelo.predict(X).tolist()
    probs = modelo.predict_proba(X)[:, 1].tolist()
    return [
        {"prediccion": 'bueno' if p == 1 else 'normal',
         "probabilidad": round(pr, 4)}
        for p, pr in zip(preds, probs)
    ]

---
## 3. Lanzar el servidor

In [ ]:
import subprocess, time, signal, os

# Lanzar uvicorn en segundo plano desde el directorio raíz del proyecto
proceso = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8000', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)  # esperar arranque

if proceso.poll() is None:
    print(f"✓ Servidor arrancado (PID={proceso.pid})")
    print("  URL: http://localhost:8000")
    print("  Docs: http://localhost:8000/docs")
else:
    stderr = proceso.stderr.read().decode()
    print(f"✗ Error al arrancar:\n{stderr}")

---
## 4. Probar la API con `requests`

In [ ]:
import requests

BASE = 'http://localhost:8000'

# GET /health
r = requests.get(f'{BASE}/health')
print(f"GET /health  →  {r.status_code}")
print(f"  {r.json()}")

# GET /model/info
r = requests.get(f'{BASE}/model/info')
print(f"\nGET /model/info  →  {r.status_code}")
info = r.json()
if 'metricas_test' in info:
    print(f"  Métricas: {info['metricas_test']}")

In [ ]:
# POST /predict — vino ejemplo
vino_ejemplo = {
    "tipo": "tinto",
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "pH": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}

r = requests.post(f'{BASE}/predict', json=vino_ejemplo)
print(f"POST /predict  →  {r.status_code}")
pred = r.json()
for k, v in pred.items():
    print(f"  {k}: {v}")

In [ ]:
# Probar validación de Pydantic — datos erróneos
vino_invalido = {
    **vino_ejemplo,
    "alcohol": 99,           # fuera de rango (>15)
    "free_sulfur_dioxide": 200,  # mayor que total (100)
    "total_sulfur_dioxide": 100
}

r = requests.post(f'{BASE}/predict', json=vino_invalido)
print(f"POST con datos inválidos  →  {r.status_code}")
print(f"  Error: {r.json()['detail'][0]['msg']}")

In [ ]:
# POST /predict/batch — múltiples vinos
import pandas as pd

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
muestra = df.head(5)

vinos_batch = []
for _, row in muestra.iterrows():
    vinos_batch.append({
        "tipo":                  row['tipo'],
        "fixed_acidity":         row['fixed acidity'],
        "volatile_acidity":      row['volatile acidity'],
        "citric_acid":           row['citric acid'],
        "residual_sugar":        row['residual sugar'],
        "chlorides":             row['chlorides'],
        "free_sulfur_dioxide":   row['free sulfur dioxide'],
        "total_sulfur_dioxide":  row['total sulfur dioxide'],
        "density":               row['density'],
        "pH":                    row['pH'],
        "sulphates":             row['sulphates'],
        "alcohol":               row['alcohol'],
    })

r = requests.post(f'{BASE}/predict/batch', json=vinos_batch)
print(f"POST /predict/batch ({len(vinos_batch)} vinos)  →  {r.status_code}")
for i, resultado in enumerate(r.json()):
    real = int(muestra.iloc[i]['buen_vino'])
    print(f"  Vino {i+1}: pred={resultado['prediccion']:6}  prob={resultado['probabilidad']:.3f}  real={'bueno' if real else 'normal'}")

In [ ]:
# Detener servidor
proceso.terminate()
proceso.wait()
print("✓ Servidor detenido")

---
## Práctica

### Ejercicio 1 — Endpoint `/predict/dl` con el modelo TensorFlow

**Enunciado:** Añade a `api/main.py` un nuevo endpoint `POST /predict/dl` que:

1. Cargue `models/dl_clf_v1.keras` y `models/dl_scaler.joblib`
2. Devuelva la predicción con `"modelo": "dl_v1"`
3. Incluya además el campo `"puntuacion_estimada"` usando `models/dl_reg_v1.keras`
4. Prueba ambos endpoints y compara resultados

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

Añade al final de `api/main.py`:

```python
from tensorflow import keras as tf_keras

def cargar_modelos_dl():
    if 'dl_clf' not in _modelos:
        _modelos['dl_clf'] = tf_keras.models.load_model('models/dl_clf_v1.keras')
        _modelos['dl_reg'] = tf_keras.models.load_model('models/dl_reg_v1.keras')
        _modelos['dl_sc']  = joblib.load('models/dl_scaler.joblib')

@app.post("/predict/dl", response_model=PrediccionSalida)
def predecir_dl(vino: VinoEntrada):
    """Predicción usando red neuronal (TensorFlow/Keras)."""
    cargar_modelos_dl()
    X_raw  = entrada_a_array(vino)                            # sin escalar
    X_sc   = _modelos['dl_sc'].transform(X_raw).astype('float32')
    prob   = float(_modelos['dl_clf'].predict(X_sc, verbose=0)[0, 0])
    punt   = float(_modelos['dl_reg'].predict(X_sc, verbose=0)[0, 0])
    return PrediccionSalida(
        prediccion          = 'bueno' if prob >= 0.5 else 'normal',
        probabilidad        = round(prob, 4),
        puntuacion_estimada = round(punt, 2),
        modelo              = 'dl_v1',
        timestamp           = datetime.now().isoformat()
    )
```

Luego prueba desde el notebook:
```python
r_dl = requests.post('http://localhost:8000/predict/dl', json=vino_ejemplo)
print(r_dl.json())
```
</details>

---
### Ejercicio 2 — Middleware de logging

**Enunciado:** FastAPI tiene un sistema de middleware que intercepta todas las peticiones. Añade uno que registre en `logs/api_requests.log` cada petición con:
- timestamp
- método (GET/POST)
- ruta (`/predict`)
- código de respuesta
- tiempo de respuesta en ms

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

Añade a `api/main.py` después de `app = FastAPI(...)`:

```python
import time, logging
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request

logging.basicConfig(
    filename='logs/api_requests.log',
    level=logging.INFO,
    format='%(message)s'
)

class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        inicio = time.perf_counter()
        respuesta = await call_next(request)
        duracion_ms = (time.perf_counter() - inicio) * 1000
        logging.info(
            f"{datetime.now().isoformat()} | {request.method} {request.url.path} "
            f"| {respuesta.status_code} | {duracion_ms:.1f}ms"
        )
        return respuesta

app.add_middleware(LoggingMiddleware)
```
</details>

---
### Ejercicio 3 — Test de carga

**Enunciado:** Simula 100 peticiones al endpoint `/predict` y analiza el rendimiento:

1. Envía 100 peticiones una a una midiendo el tiempo de cada una
2. Calcula: media, mediana, percentil 95 y máximo de tiempo de respuesta
3. ¿Cuántas peticiones por segundo aguanta la API?
4. Usa `concurrent.futures.ThreadPoolExecutor` para enviar 10 en paralelo y compara

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import requests, time, numpy as np
from concurrent.futures import ThreadPoolExecutor

URL = 'http://localhost:8000/predict'

def enviar_peticion(_):
    t0 = time.perf_counter()
    r = requests.post(URL, json=vino_ejemplo)
    return (time.perf_counter() - t0) * 1000, r.status_code

# Serie
print("Serie (100 peticiones):")
t0 = time.perf_counter()
tiempos = [enviar_peticion(None)[0] for _ in range(100)]
total = time.perf_counter() - t0
tiempos = np.array(tiempos)
print(f"  Tiempo total: {total:.1f}s  ({100/total:.1f} req/s)")
print(f"  Media: {tiempos.mean():.1f}ms  Mediana: {np.median(tiempos):.1f}ms  P95: {np.percentile(tiempos,95):.1f}ms  Máx: {tiempos.max():.1f}ms")

# Paralelo
print("\nParalelo (10 workers, 100 peticiones):")
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as pool:
    resultados = list(pool.map(enviar_peticion, range(100)))
total_p = time.perf_counter() - t0
tiempos_p = np.array([r[0] for r in resultados])
print(f"  Tiempo total: {total_p:.1f}s  ({100/total_p:.1f} req/s)")
print(f"  Media: {tiempos_p.mean():.1f}ms  P95: {np.percentile(tiempos_p,95):.1f}ms")
```
</details>

---
## Resumen

```python
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Mi API")

class Entrada(BaseModel):
    valor: float = Field(..., ge=0, le=100, description="Valor entre 0 y 100")

class Salida(BaseModel):
    resultado: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=Salida)
def predecir(datos: Entrada):
    return Salida(resultado="positivo", score=datos.valor / 100)

# Lanzar: uvicorn api.main:app --reload --port 8000
```

## ¿Qué viene en la próxima clase?
En la **Clase 7** documentaremos el proyecto: docstrings, README, OpenAPI/Swagger y flujo básico de Git/GitHub.